In [ ]:
import pandas as pd, numpy as np
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import GroupShuffleSplit

ds = fetch_ucirepo(id=967)
X = ds.data.features.copy()
y = 1 - ds.data.targets.iloc[:, 0] 
groups = X["Domain"]                
X = X.select_dtypes(include="number")   

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr, te = next(gss.split(X, y, groups))
Xtr, Xte, ytr, yte = X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te]

print("Total:", X.shape, "| Train:", Xtr.shape, "| Test:", Xte.shape)
print("Phishing share in test:", round(yte.mean(), 3))

Total: (235795, 50) | Train: (189074, 50) | Test: (46721, 50)
Phishing share in test: 0.424


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

models = {
 "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
 "Decision Tree": DecisionTreeClassifier(random_state=42),
 "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
 "XGBoost": XGBClassifier(n_estimators=300, eval_metric="logloss", random_state=42),
}
rows = []
for name, m in models.items():
    m.fit(Xtr, ytr)
    p = m.predict_proba(Xte)[:, 1]
    pred = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
    rows.append({"Model": name, "Accuracy": accuracy_score(yte, pred),
        "Precision": precision_score(yte, pred), "Recall": recall_score(yte, pred),
        "F1": f1_score(yte, pred), "FPR": fp / (fp + tn), "ROC-AUC": roc_auc_score(yte, p)})
results = pd.DataFrame(rows).round(4)
results

,Model,Accuracy,Precision,Recall,F1,FPR,ROC-AUC
0,Logistic Regression,0.9999,1.0,0.9997,0.9999,0.0,1.0000
1,Decision Tree,1.0000,1.0,0.9999,0.9999,0.0,0.9999
2,Random Forest,1.0000,1.0,0.9999,0.9999,0.0,1.0000
3,XGBoost,1.0000,1.0,0.9999,0.9999,0.0,1.0000


In [ ]:
import sys
from pathlib import Path
sys.path.extend(["src", "../src"])
import importlib, features; importlib.reload(features)
from features import url_features, homepage
import joblib
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from xgboost import XGBClassifier

root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
raw = ds.data.features
rows, keep = [], []
for i, u in enumerate(raw["URL"]):
    try:
        rows.append(url_features(homepage(u))); keep.append(i)
    except Exception:
        pass
U = pd.DataFrame(rows)
yy = (1 - ds.data.targets.iloc[:, 0]).values[keep]
gg = raw["Domain"].values[keep]

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(U, yy, gg))
clf = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, eval_metric="logloss", random_state=42)
clf.fit(U.iloc[tr], yy[tr])
p = clf.predict_proba(U.iloc[te])[:, 1]; pred = (p >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(yy[te], pred).ravel()
print("Accuracy", round(accuracy_score(yy[te], pred), 4), "| Precision", round(precision_score(yy[te], pred), 4),
      "| Recall", round(recall_score(yy[te], pred), 4), "| F1", round(f1_score(yy[te], pred), 4), "| FPR", round(fp / (fp + tn), 4))

(root / "models").mkdir(exist_ok=True)
joblib.dump({"model": clf, "columns": list(U.columns)}, root / "models" / "url_model.joblib")
print("Model saved")

Accuracy 0.938 | Precision 0.9707 | Recall 0.8803 | F1 0.9233 | FPR 0.0196
Model saved


In [ ]:
import sys
sys.path.extend(["src", "../src"])
import importlib, detector
importlib.reload(detector)

for u in ["https://www.google.com", "https://www.wikipedia.org", "http://secure-login-sbi.verify-account.xyz/update"]:
    r = detector.check(u)
    print(u)
    print("   Risk:", round(r["risk"], 1), "% | Level:", r["level"])
    for t in r["reasons"]:
        print("   -", t)

https://www.google.com
   Risk: 0.2 % | Level: LOW
https://www.wikipedia.org
   Risk: 0.4 % | Level: LOW
http://secure-login-sbi.verify-account.xyz/update
   Risk: 100.0 % | Level: HIGH
   - The site does not use HTTPS.
   - The URL is unusually long (49 characters).
   - It contains 5 sensitive words such as login, verify, secure or bank.


In [ ]:
import shap
xgb = models["XGBoost"]
i = int(np.flatnonzero(yte.values == 1)[0])      
sv = shap.TreeExplainer(xgb).shap_values(Xte.iloc[[i]])[0]
top = pd.Series(sv, index=Xte.columns).sort_values(ascending=False).head(4)
print("Risk %:", round(xgb.predict_proba(Xte.iloc[[i]])[0, 1] * 100, 1))
print(top)

Risk %: 100.0
URLSimilarityIndex    6.854701
LineOfCode            3.801434
NoOfImage             0.680646
NoOfSelfRef           0.667046
dtype: float32


In [ ]:
import sys
sys.path.extend(["src", "../src"])
import importlib, features, detector
importlib.reload(features); importlib.reload(detector)
import pandas as pd

urls = ["https://www.youtube.com/watch?v=T4lGm7MjA6Y",
        "https://internshala.com/student/dashboard",
        "https://www.geeksforgeeks.org/python-programming-language/"]
for u in urls:
    r = detector.check(u)          # same path as the app
    f = r["features"]
    x = pd.DataFrame([f])[detector.COLS]
    sv = pd.Series(detector.EXPLAINER.shap_values(x)[0], index=detector.COLS)
    print(u, "-> app risk", round(r["risk"], 1), r["level"])
    print("  pushes UP  :", sv.sort_values(ascending=False).head(3).round(2).to_dict())
    print("  values     :", {k: f[k] for k in ["url_length", "num_subdomains", "num_hyphens", "path_depth", "num_digits", "url_entropy"]})

https://www.youtube.com/watch?v=T4lGm7MjA6Y -> app risk 2.8 LOW
  pushes UP  : {'num_dots': 0.09000000357627869, 'path_depth': 0.0, 'is_ip_host': 0.0}
  values     : {'url_length': 23, 'num_subdomains': 1, 'num_hyphens': 0, 'path_depth': 0, 'num_digits': 0, 'url_entropy': 3.762}
https://internshala.com/student/dashboard -> app risk 100.0 HIGH
  pushes UP  : {'num_subdomains': 7.599999904632568, 'num_dots': 3.200000047683716, 'has_at_symbol': 0.0}
  values     : {'url_length': 23, 'num_subdomains': 0, 'num_hyphens': 0, 'path_depth': 0, 'num_digits': 0, 'url_entropy': 3.882}
https://www.geeksforgeeks.org/python-programming-language/ -> app risk 4.7 LOW
  pushes UP  : {'hostname_length': 0.10999999940395355, 'num_dots': 0.07000000029802322, 'is_ip_host': 0.0}
  values     : {'url_length': 29, 'num_subdomains': 1, 'num_hyphens': 0, 'path_depth': 0, 'num_digits': 0, 'url_entropy': 3.676}


In [ ]:
from sklearn.metrics import roc_auc_score
single = pd.Series({c: roc_auc_score(ytr, Xtr[c]) for c in Xtr.columns})
single = (single - 0.5).abs() + 0.5          
single = single.sort_values(ascending=False)
print(single.head(10).round(4))

URLSimilarityIndex    0.9961
LineOfCode            0.9899
NoOfExternalRef       0.9878
NoOfImage             0.9793
NoOfSelfRef           0.9721
NoOfJS                0.9702
NoOfCSS               0.9572
HasSocialNet          0.8950
HasCopyrightInfo      0.8748
HasDescription        0.8463
dtype: float64


In [ ]:
leaky = single[single > 0.97].index.tolist()
print("Removed features:", leaky)
Xtr2, Xte2 = Xtr.drop(columns=leaky), Xte.drop(columns=leaky)
rows = []
for name in ["Logistic Regression", "Random Forest", "XGBoost"]:
    m = models[name]
    m.fit(Xtr2, ytr)
    p = m.predict_proba(Xte2)[:, 1]
    pred = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
    rows.append({"Model": name, "Accuracy": accuracy_score(yte, pred), "Precision": precision_score(yte, pred),
        "Recall": recall_score(yte, pred), "F1": f1_score(yte, pred), "FPR": fp/(fp+tn), "ROC-AUC": roc_auc_score(yte, p)})
pd.DataFrame(rows).round(4)

Removed features: ['URLSimilarityIndex', 'LineOfCode', 'NoOfExternalRef', 'NoOfImage', 'NoOfSelfRef', 'NoOfJS']


,Model,Accuracy,Precision,Recall,F1,FPR,ROC-AUC
0,Logistic Regression,0.9990,0.9989,0.9987,0.9988,0.0008,1.0
1,Random Forest,0.9997,0.9996,0.9996,0.9996,0.0003,1.0
2,XGBoost,0.9997,0.9997,0.9997,0.9997,0.0002,1.0


In [ ]:
import sys
sys.path.extend(["src", "../src"])
from features import url_features

print(url_features("https://www.google.com"))
print(url_features("http://secure-login-sbi.verify-account.xyz/update"))

{'url_length': 22, 'hostname_length': 14, 'num_dots': 2, 'num_hyphens': 0, 'num_digits': 0, 'has_at_symbol': 0, 'is_ip_host': 0, 'num_subdomains': 1, 'path_depth': 0, 'url_entropy': 3.664, 'keyword_count': 0, 'uses_https': 1}
{'url_length': 49, 'hostname_length': 35, 'num_dots': 2, 'num_hyphens': 3, 'num_digits': 0, 'has_at_symbol': 0, 'is_ip_host': 0, 'num_subdomains': 1, 'path_depth': 1, 'url_entropy': 4.476, 'keyword_count': 5, 'uses_https': 0}


In [ ]:
import importlib, features
importlib.reload(features)
import pandas as pd

print(pd.Series(features.extract_features("https://www.wikipedia.org")))

url_length                25.000
hostname_length           17.000
num_dots                   2.000
num_hyphens                0.000
num_digits                 0.000
has_at_symbol              0.000
is_ip_host                 0.000
num_subdomains             1.000
path_depth                 0.000
url_entropy                3.814
keyword_count              0.000
uses_https                 1.000
domain_age_days         9446.000
days_to_expiry            49.000
whois_available            1.000
cert_valid                 1.000
cert_age_days             50.000
fetch_ok                   1.000
num_redirects              0.000
has_password_field         0.000
num_forms                  1.000
form_external              0.000
num_iframes                0.000
external_link_ratio        0.055
empty_link_ratio           0.010
num_external_scripts       0.000
dtype: float64
